In [139]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [140]:
import pandas as pd
df_cleaned_resume = pd.read_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/resume_cleaned.pkl')
df_cleaned_jd = pd.read_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/jd_cleaned.pkl')

In [141]:
df_cleaned_resume

,ID,Resume_str,Resume_html,Category,cleaned_resume
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr administrator marketing associate hr admini...
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR,hr specialist hr operation summary versatile m...
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr director summary 20 year experience recruit...
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr specialist summary dedicate driven dynamic ...
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR,hr manager skill highlight hr skill hr departm...
...,...,...,...,...,...
1803,31694970,INDUSTRIAL ENGINEERING INTERN P...,"<div class=""fontsize fontface vmargins hmargin...",ENGINEERING,industrial engineering intern profile outstand...
1804,33685075,"MANAGER, QUALITY ENGINEERING ...","<div class=""fontsize fontface vmargins hmargin...",ENGINEERING,manager quality engineering executive summary ...
1805,77828437,MECHANICAL DESIGN ENGINEERING INTERN ...,"<div class=""fontsize fontface vmargins hmargin...",ENGINEERING,mechanical design engineering intern summary s...
1806,35172961,PROCESS ENGINEERING INTERN ...,"<div class=""fontsize fontface vmargins hmargin...",ENGINEERING,process engineering intern profile phd cleanro...


In [142]:
df_cleaned_jd

,category,job_title,job_description,cleaned_description
0,INFORMATION-TECHNOLOGY,IT Support Technician Job in Madison,TeamSoft is seeing an IT Support Specialist to...,teamsoft see support specialist join client ma...
1,FINANCE,Senior Accountant/Analyst Job in Denver,Would you like to grow your accounting and fin...,like grow accounting finance career great star...
2,ENGINEERING,Quality Engineer Job in Durham,Experis Engineering has an immediate opening f...,experis engineering immediate opening quality ...
3,SALES,Sales Professional Job in Las Vegas,Aflac Insurance Sales Agent While a career in ...,aflac insurance sale agent career sale success...
4,HR,Human Resources Manager Job in Dallas,"Human Resource Manager Salary $55,000 - $75,0...","human resource manager salary $ 55,000 $ 75,00..."
5,HEALTHCARE,Registered Nurse - Clinic Job in Houston,Job Description Position Summary: Provides pro...,job description position summary provide profe...


In [143]:
!python -m spacy download en_core_web_sm
import spacy
nlp = spacy.load('en_core_web_sm')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 34.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [144]:
#checking if spacy's en_core_web_sm have in-built name entity recognition
doc = nlp('I have experience with Python, SQL, and project management.')
for ent in doc.ents:
  print(ent.text, ent.label_)

SQL ORG


As there is no built in function in spacy's  en_core_web_sm  library to extract skills, so we give it custom skill set to detect, by using a well known technique called EntityRuler()

In [145]:
from sklearn.feature_extraction.text import CountVectorizer

categories = ['INFORMATION-TECHNOLOGY', 'ENGINEERING', 'FINANCE', 'SALES', 'HR', 'HEALTHCARE']

for cat in categories:
  texts = df_cleaned_resume[df_cleaned_resume['Category']== cat]['cleaned_resume']
  vectorizer = CountVectorizer(ngram_range=(1,2), max_features=20)
  x = vectorizer.fit_transform(texts)
  print(cat, ' :', vectorizer.get_feature_names_out())
  print()



INFORMATION-TECHNOLOGY  : ['application' 'business' 'database' 'datum' 'information'
 'information technology' 'manage' 'management' 'microsoft' 'network'
 'project' 'provide' 'security' 'server' 'service' 'software' 'support'
 'system' 'team' 'technology']

ENGINEERING  : ['customer' 'design' 'develop' 'development' 'engineer' 'engineering'
 'equipment' 'experience' 'management' 'new' 'process' 'product' 'project'
 'quality' 'service' 'skill' 'software' 'system' 'team' 'test']

FINANCE  : ['account' 'accounting' 'analysis' 'budget' 'business' 'customer'
 'finance' 'financial' 'manage' 'management' 'manager' 'monthly' 'prepare'
 'process' 'project' 'report' 'sale' 'skill' 'system' 'team']

SALES  : ['associate' 'business' 'client' 'customer' 'customer service'
 'experience' 'high' 'maintain' 'management' 'manager' 'need' 'new'
 'product' 'provide' 'sale' 'service' 'skill' 'store' 'team' 'training']

HR  : ['benefit' 'business' 'development' 'employee' 'hr' 'human'
 'human resource' 'ma



> 'accuracy was low earlier.'


Problem :- our company names and some other strings are occuring rarely and as per our spacy skill extracter (entityruler()), which was not able to remove these words and our mathematical model (cosine similarity) giving higher weightage to these words.

solution:- spacy able to recognise organisation names so we will use that to remove company names

In [146]:
def remove_org_name(text):
  doc = nlp(text)
  org_spans = [(ent.start_char, ent.end_char) for ent in doc.ents if ent.label_ == 'ORG']

  if not org_spans:
    return text

  cleaned = text
  for start,end in sorted(org_spans, reverse = True):
    cleaned = cleaned[:start]+cleaned[end:]

  return cleaned

Removed Generic terms [management, business, team, process, new, provide, experience, system] as they are occuring in every sector and they will not useful to differentiate one's skills to another's skills.

In [147]:
skill_list = {
    'INFORMATION-TECHNOLOGY': ['database', 'network', 'server', 'security', 'software', 'microsoft', 'information technology'],
    'ENGINEERING': ['engineering', 'design', 'development', 'quality', 'test', 'product'],
    'FINANCE': ['accounting', 'financial', 'budget', 'analysis', 'account'],
    'SALES': ['customer service', 'sale', 'client', 'store', 'training'],
    'HR': ['human resource', 'payroll', 'benefit', 'policy', 'employee'],
    'HEALTHCARE': ['patient', 'medical', 'healthcare', 'care', 'health']
}

In [148]:
all_skills = set()
for skills in skill_list.values():
  all_skills.update(skills)

patterns = [{'label': 'skill', 'pattern': skill} for skill in all_skills]

ruler = nlp.add_pipe('entity_ruler', before='ner')
ruler.add_patterns(patterns)

In [149]:
#testing our EntityRuler NER
doc = nlp("I have experience in database management, network security, and customer service.")
for ent in doc.ents:
  print(ent.text, ent.label_)

database skill
network skill
security skill
customer service skill


In [150]:
#form the function which extract skills from our resume
def extract_skills(text):
  doc = nlp(text)
  skills = [ent.text for ent in doc.ents if ent.label_ == 'skill']
  return list(set(skills))

In [151]:
df_cleaned_resume['extracted_skills'] = df_cleaned_resume['cleaned_resume'].apply(extract_skills)
df_cleaned_jd['extracted_skills'] = df_cleaned_jd['cleaned_description'].apply(extract_skills)

print(df_cleaned_resume[['Category', 'extracted_skills']].head(10))
print(df_cleaned_jd[['category', 'extracted_skills']])

  Category                                   extracted_skills
0       HR  [payroll, health, test, training, server, anal...
1       HR  [quality, employee, product, design, training,...
2       HR  [quality, employee, health, development, desig...
3       HR  [quality, training, customer service, microsof...
4       HR  [quality, employee, health, development, train...
5       HR  [quality, employee, design, training, developm...
6       HR  [quality, employee, health, development, train...
7       HR  [quality, employee, health, development, train...
8       HR  [employee, development, training, customer ser...
9       HR  [employee, benefit, development, training, acc...
                 category                                   extracted_skills
0  INFORMATION-TECHNOLOGY  [development, design, analysis, account, micro...
1                 FINANCE  [product, analysis, account, accounting, finan...
2             ENGINEERING  [quality, health, test, engineering, product, ...
3         

In [152]:
sample_hr_resume = df_cleaned_resume[df_cleaned_resume['Category'] == 'HR']['cleaned_resume'].iloc[1]
check = ['quality', 'software', 'policy', 'design', 'development']

from collections import Counter
words = sample_hr_resume[:800].split()
counts = Counter(words)
for word in check:
  print(f"{word} appears {counts[word]} times")


quality appears 0 times
software appears 0 times
policy appears 1 times
design appears 1 times
development appears 0 times


In [153]:
custom_stopwords = {'include', 'city', 'company', 'work', 'state'}

def clean_text(text):
  doc = nlp(text.lower())
  tokens = [
      token.lemma_
      for token in doc
      if not token.is_stop
      and not token.is_punct
      and not token.is_space
      and token.lemma_.strip() != ''
      and token.lemma_ not in custom_stopwords
  ]

  return ' '.join(tokens)

In [154]:
def cleaned_jd_pipeline(text):
  no_org_text = remove_org_name(text)
  return clean_text(no_org_text)

df_cleaned_jd['cleaned_description'] = df_cleaned_jd['job_description'].apply(cleaned_jd_pipeline)
print(df_cleaned_jd[['cleaned_description','job_description']])

                                 cleaned_description  \
0  see support specialist join client madison ide...   
1  like grow accounting finance career great star...   
2  experis engineering immediate opening quality ...   
3  career sale successful rewarding venture life ...   
4  human resource manager salary $ 55,000 $ 75,00...   
5  job description position summary provide profe...   

                                     job_description  
0  TeamSoft is seeing an IT Support Specialist to...  
1  Would you like to grow your accounting and fin...  
2  Experis Engineering has an immediate opening f...  
3  Aflac Insurance Sales Agent While a career in ...  
4  Human Resource Manager  Salary $55,000 - $75,0...  
5  Job Description Position Summary: Provides pro...  


In [155]:
print("teamsoft still present:", 'teamsoft' in df_cleaned_jd[df_cleaned_jd['category']=='INFORMATION-TECHNOLOGY']['cleaned_description'].iloc[0])
print("aflac still present:", 'aflac' in df_cleaned_jd[df_cleaned_jd['category']=='SALES']['cleaned_description'].iloc[0])


teamsoft still present: True
aflac still present: True


In [156]:
from sklearn.feature_extraction.text import TfidfVectorizer

all_texts = list(df_cleaned_resume['cleaned_resume']) + list(df_cleaned_jd['cleaned_description'])
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(all_texts)

In [157]:
print(tfidf_matrix.shape)

(703, 17136)


In [158]:
print(tfidf_matrix[:10, :10].toarray())

[[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.04345251 0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.

In [159]:
num_resumes = df_cleaned_resume.shape[0]
resume_vectors = tfidf_matrix[:num_resumes]
jd_vectors = tfidf_matrix[num_resumes:]


print(resume_vectors.shape)
print(jd_vectors.shape)

(697, 17136)
(6, 17136)


In [160]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(resume_vectors, jd_vectors)
print(similarity_matrix.shape)

(697, 6)


In [161]:
print(similarity_matrix)

[[0.04810346 0.06250257 0.055381   0.10244972 0.08105191 0.04565973]
 [0.0436544  0.05078145 0.10209577 0.06435448 0.09000305 0.04596838]
 [0.08228704 0.05073942 0.06196083 0.095364   0.14315012 0.06093925]
 ...
 [0.02449801 0.01602976 0.09582703 0.02033288 0.03376689 0.05546926]
 [0.01582061 0.00819284 0.05138346 0.02077143 0.01470102 0.01057604]
 [0.04851061 0.02296338 0.21028638 0.04269504 0.04834374 0.04763549]]


In [162]:
import numpy as np

category_order = list(df_cleaned_jd['category'])
df_cleaned_resume['best_match_category'] = [category_order[i] for i in similarity_matrix.argmax(axis=1)]
df_cleaned_resume['best_match_score'] = similarity_matrix.max(axis=1)


accuracy = (df_cleaned_resume['Category'] == df_cleaned_resume['best_match_category']).mean()
print(f"accuracy: {accuracy:.2%}")

print(df_cleaned_resume[['Category', 'best_match_category', 'best_match_score']].sample(10))

accuracy: 62.70%
                    Category     best_match_category  best_match_score
1590                 FINANCE                 FINANCE          0.145695
1014                   SALES                   SALES          0.103024
287   INFORMATION-TECHNOLOGY                 FINANCE          0.065350
684               HEALTHCARE                   SALES          0.094800
302   INFORMATION-TECHNOLOGY                 FINANCE          0.090576
247   INFORMATION-TECHNOLOGY                      HR          0.056229
1526                 FINANCE  INFORMATION-TECHNOLOGY          0.084359
252   INFORMATION-TECHNOLOGY                 FINANCE          0.069401
1020                   SALES                   SALES          0.053456
1564                 FINANCE                 FINANCE          0.087551


In [163]:
confusion = pd.crosstab(df_cleaned_resume['Category'], df_cleaned_resume['best_match_category'])
print(confusion)

best_match_category     ENGINEERING  FINANCE  HEALTHCARE  HR  \
Category                                                       
ENGINEERING                      96        2           2   6   
FINANCE                           1      101           2   4   
HEALTHCARE                        9        6          55  11   
HR                                1        2           3  94   
INFORMATION-TECHNOLOGY           32        8           8  25   
SALES                            28        2          10   5   

best_match_category     INFORMATION-TECHNOLOGY  SALES  
Category                                               
ENGINEERING                                  5      7  
FINANCE                                      1      9  
HEALTHCARE                                   4     30  
HR                                           1      9  
INFORMATION-TECHNOLOGY                      26     21  
SALES                                        6     65  


In [164]:
sample_it_resume_idx = df_cleaned_resume[df_cleaned_resume['Category'] =='INFORMATION-TECHNOLOGY'].index[0]
position = df_cleaned_resume.index.get_loc(sample_it_resume_idx)

print('similarity of infotech to each JD :')
for i, cat in enumerate(category_order):
  print(cat, ':', similarity_matrix[position][i])

similarity of infotech to each JD :
INFORMATION-TECHNOLOGY : 0.06518317906428332
FINANCE : 0.030308118065853238
ENGINEERING : 0.060167131376906845
SALES : 0.05388960721510883
HR : 0.052972042649797484
HEALTHCARE : 0.021625557698718014


In [165]:
df_cleaned_resume.to_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/df_cleaned_resume.pkl')
df_cleaned_jd.to_pickle('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/df_cleaned_jd.pkl')

import pickle
with open('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

import numpy as np
np.save('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/resume_vectors.npy', resume_vectors.toarray())
np.save('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/jd_vectors.npy', jd_vectors.toarray())
np.save('/content/drive/MyDrive/Colab Notebooks/Future_ML_03/data/similarity_matrix.npy', similarity_matrix)

print("All saved.")

All saved.
